# 🏦 Deal Desk Agent — Full Deployment Notebook

**Anthropic + Google Cloud: Better Together**

This notebook deploys the complete Deal Desk Agent solution from scratch:

| Component | What It Does | GCP Service |
|-----------|-------------|-------------|
| **BigQuery** | Client data, compliance records, market intelligence | BigQuery |
| **Backend** | Conversational AI, ADK pipeline, A2A handler | Cloud Run |
| **Frontend** | React chat UI with agent feed + noVNC panel | Cloud Run |
| **Browser VM** | Computer Use: Claude drives Salesforce via Chrome | Compute Engine |
| **Static IP** | Persistent IP for the browser VM | VPC Network |
| **Firewall** | Open ports 6080 (noVNC) + 8090 (agent server) | VPC Firewall |

> **Author:** Schneider Larbi, Sr. Manager, Global Partner Technical Architecture AI & SaaS ISV, Google Cloud

> **Models:** Claude Opus 4.5, Sonnet 4.6, Haiku 4.5 on Vertex AI

---

## 1. Configuration & Parameters

**Edit the values below** to match your GCP environment. All subsequent cells reference these variables.

In [ ]:
#@title 1.1 — Project Configuration { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════
#  EDIT THESE VALUES FOR YOUR ENVIRONMENT
# ═══════════════════════════════════════════════════════════════

PROJECT_ID        = "your-gcp-project-id"       #@param {type:"string"}
REGION            = "us-east5"                   #@param {type:"string"}
INFRA_REGION      = "us-central1"                #@param {type:"string"}
ZONE              = "us-central1-a"              #@param {type:"string"}
BQ_DATASET        = "deal_desk_agent"            #@param {type:"string"}

# Claude Models (must be enabled on your project in REGION)
OPUS_MODEL        = "claude-opus-4-5@20251101"   #@param {type:"string"}
SONNET_MODEL      = "claude-sonnet-4-6@default"  #@param {type:"string"}
HAIKU_MODEL       = "claude-haiku-4-5@20251001"  #@param {type:"string"}

# Salesforce
SALESFORCE_URL    = "https://your-org.lightning.force.com"  #@param {type:"string"}

# Artifact Registry
AR_REPO           = "deal-desk-agent"            #@param {type:"string"}

# Derived values (do not edit)
AR_PREFIX = f"{INFRA_REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}"

print("═" * 60)
print("  Deal Desk Agent — Configuration")
print("═" * 60)
print(f"  Project:     {PROJECT_ID}")
print(f"  Model Region:{REGION}")
print(f"  Infra Region:{INFRA_REGION}")
print(f"  Zone:        {ZONE}")
print(f"  BQ Dataset:  {BQ_DATASET}")
print(f"  AR Prefix:   {AR_PREFIX}")
print(f"  Salesforce:  {SALESFORCE_URL}")
print("═" * 60)


In [ ]:
#@title 1.2 — Authenticate & Set Project

from google.colab import auth
auth.authenticate_user()

!gcloud config set project {PROJECT_ID}
print(f"\n✅ Authenticated and project set to {PROJECT_ID}")


## 2. Enable APIs & Create Resources

Enables all required GCP APIs and creates the Artifact Registry repository.

In [ ]:
#@title 2.1 — Enable APIs

APIS = [
    "aiplatform.googleapis.com",
    "bigquery.googleapis.com",
    "run.googleapis.com",
    "compute.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "discoveryengine.googleapis.com",
    "secretmanager.googleapis.com",
]

for api in APIS:
    !gcloud services enable {api} --quiet
    print(f"  ✅ {api}")

print(f"\n✅ All {len(APIS)} APIs enabled")


In [ ]:
#@title 2.2 — Create Artifact Registry Repository

!gcloud artifacts repositories create {AR_REPO} \
    --repository-format=docker \
    --location={INFRA_REGION} \
    --quiet 2>/dev/null || echo 'Repository already exists'

print(f"✅ Artifact Registry: {AR_PREFIX}")


In [ ]:
#@title 2.3 — Create Service Account & IAM Bindings

SA_NAME = "deal-desk-agent-sa"
SA_EMAIL = f"{SA_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

!gcloud iam service-accounts create {SA_NAME} \
    --display-name="Deal Desk Agent" \
    --quiet 2>/dev/null || echo 'SA already exists'

ROLES = [
    "roles/aiplatform.user",
    "roles/bigquery.dataEditor",
    "roles/bigquery.jobUser",
]

for role in ROLES:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member="serviceAccount:{SA_EMAIL}" \
        --role={role} --condition=None --quiet 2>/dev/null

print(f"\n✅ Service account: {SA_EMAIL}")
print("   Roles: aiplatform.user, bigquery.dataEditor, bigquery.jobUser")


## 3. BigQuery — Dataset, Tables & Sample Data

Creates the `deal_desk_agent` dataset with 4 tables and populates them with realistic FSI sample data (10 clients, 10 compliance records, 15 market intelligence entries).

In [ ]:
#@title 3.1 — Create Dataset

from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.DatasetReference(PROJECT_ID, BQ_DATASET)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = "US"

try:
    bq.create_dataset(dataset)
    print(f"✅ Dataset created: {PROJECT_ID}.{BQ_DATASET}")
except Exception as e:
    if "Already Exists" in str(e):
        print(f"✅ Dataset already exists: {PROJECT_ID}.{BQ_DATASET}")
    else:
        raise


In [ ]:
#@title 3.2 — Create Tables

SCHEMAS = {
    "clients": [
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("aum_millions", "FLOAT64"),
        bigquery.SchemaField("strategy", "STRING"),
        bigquery.SchemaField("domicile", "STRING"),
        bigquery.SchemaField("fee_structure", "STRING"),
        bigquery.SchemaField("relationship_status", "STRING"),
        bigquery.SchemaField("primary_contact", "STRING"),
        bigquery.SchemaField("primary_contact_title", "STRING"),
        bigquery.SchemaField("onboarded_date", "DATE"),
    ],
    "compliance_records": [
        bigquery.SchemaField("client_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("kyc_status", "STRING"),
        bigquery.SchemaField("aml_status", "STRING"),
        bigquery.SchemaField("sanctions_clear", "BOOL"),
        bigquery.SchemaField("finra_registered", "BOOL"),
        bigquery.SchemaField("risk_tier", "STRING"),
        bigquery.SchemaField("last_review_date", "DATE"),
        bigquery.SchemaField("next_review_date", "DATE"),
        bigquery.SchemaField("notes", "STRING"),
    ],
    "market_intelligence": [
        bigquery.SchemaField("client_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("source_type", "STRING"),
        bigquery.SchemaField("title", "STRING"),
        bigquery.SchemaField("summary", "STRING"),
        bigquery.SchemaField("relevance_score", "FLOAT64"),
        bigquery.SchemaField("date", "DATE"),
    ],
    "deal_packages": [
        bigquery.SchemaField("deal_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("client_name", "STRING"),
        bigquery.SchemaField("aum_millions", "FLOAT64"),
        bigquery.SchemaField("strategy", "STRING"),
        bigquery.SchemaField("mandate_type", "STRING"),
        bigquery.SchemaField("fee_structure", "STRING"),
        bigquery.SchemaField("compliance_status", "STRING"),
        bigquery.SchemaField("risk_tier", "STRING"),
        bigquery.SchemaField("risk_score", "FLOAT64"),
        bigquery.SchemaField("primary_contact", "STRING"),
        bigquery.SchemaField("primary_contact_title", "STRING"),
        bigquery.SchemaField("salesforce_opportunity_id", "STRING"),
        bigquery.SchemaField("status", "STRING"),
        bigquery.SchemaField("created_by", "STRING"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
        bigquery.SchemaField("notes", "STRING"),
    ],
}

for table_name, schema in SCHEMAS.items():
    table_id = f"{PROJECT_ID}.{BQ_DATASET}.{table_name}"
    table = bigquery.Table(table_id, schema=schema)
    try:
        bq.create_table(table)
        print(f"  ✅ Created {table_name}")
    except Exception as e:
        if "Already Exists" in str(e):
            print(f"  ✅ {table_name} already exists")
        else:
            raise

print(f"\n✅ All 4 tables ready in {BQ_DATASET}")


In [ ]:
#@title 3.3 — Populate Clients Table (10 Institutional Clients)

CLIENTS = [
    {"name": "Atlas Global Advisors", "aum_millions": 1200.0, "strategy": "Multi-Strategy", "domicile": "United States", "fee_structure": "1.5/20", "relationship_status": "Active", "primary_contact": "Michael Torres", "primary_contact_title": "Managing Director", "onboarded_date": "2019-03-15"},
    {"name": "Northstar Capital Partners", "aum_millions": 750.0, "strategy": "Private Equity", "domicile": "Canada", "fee_structure": "1.5/15", "relationship_status": "Active", "primary_contact": "Catherine Dubois", "primary_contact_title": "Senior Partner", "onboarded_date": "2020-01-10"},
    {"name": "Beacon Hedge Fund", "aum_millions": 500.0, "strategy": "Global Macro", "domicile": "Cayman Islands", "fee_structure": "2/20", "relationship_status": "Active", "primary_contact": "Elena Vasquez", "primary_contact_title": "CIO", "onboarded_date": "2020-07-22"},
    {"name": "Zenith Quantitative Research", "aum_millions": 420.0, "strategy": "Quantitative Equity", "domicile": "United Kingdom", "fee_structure": "2/25", "relationship_status": "Active", "primary_contact": "James Wright", "primary_contact_title": "Head of Strategy", "onboarded_date": "2021-02-28"},
    {"name": "Meridian Investment Group", "aum_millions": 350.0, "strategy": "Event Driven", "domicile": "United States", "fee_structure": "1.75/20", "relationship_status": "Active", "primary_contact": "Robert Kim", "primary_contact_title": "Portfolio Manager", "onboarded_date": "2021-06-15"},
    {"name": "Vanguard Alpha Partners", "aum_millions": 280.0, "strategy": "Long/Short Equity", "domicile": "Singapore", "fee_structure": "2/20", "relationship_status": "Active", "primary_contact": "Wei Lin", "primary_contact_title": "CEO", "onboarded_date": "2022-01-20"},
    {"name": "Horizon Macro Fund", "aum_millions": 200.0, "strategy": "Global Macro", "domicile": "Switzerland", "fee_structure": "1.5/15", "relationship_status": "Active", "primary_contact": "Hans Mueller", "primary_contact_title": "Head of Trading", "onboarded_date": "2022-09-01"},
    {"name": "ACME Capital Management", "aum_millions": 250.0, "strategy": "Long/Short Equity", "domicile": "United States", "fee_structure": "2/20", "relationship_status": "Returning", "primary_contact": "Sarah Chen", "primary_contact_title": "CIO", "onboarded_date": "2023-11-05"},
    {"name": "Quantum Strategies LLC", "aum_millions": 95.0, "strategy": "Systematic Futures", "domicile": "United States", "fee_structure": "2/20", "relationship_status": "Prospect", "primary_contact": "Andrei Volkov", "primary_contact_title": "Head of Trading", "onboarded_date": None},
    {"name": "Pinnacle Investment Group", "aum_millions": 75.0, "strategy": "Quantitative Equity", "domicile": "United States", "fee_structure": "2.5/25", "relationship_status": "Prospect", "primary_contact": "Emily Zhao", "primary_contact_title": "Portfolio Manager", "onboarded_date": None},
]

table_id = f'{PROJECT_ID}.{BQ_DATASET}.clients'
errors = bq.insert_rows_json(table_id, CLIENTS)
if errors:
    print(f"❌ Errors: {errors}")
else:
    print(f"✅ Inserted {len(CLIENTS)} clients")


In [ ]:
#@title 3.4 — Populate Compliance Records (10 Records)

COMPLIANCE = [
    {"client_name": "Atlas Global Advisors", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": True, "risk_tier": "LOW", "last_review_date": "2025-01-15", "next_review_date": "2026-01-15", "notes": "Clean record. Annual review on schedule."},
    {"client_name": "Northstar Capital Partners", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": False, "risk_tier": "LOW", "last_review_date": "2025-02-20", "next_review_date": "2026-02-20", "notes": "Canadian entity. FINRA N/A."},
    {"client_name": "Beacon Hedge Fund", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": True, "risk_tier": "MEDIUM", "last_review_date": "2025-03-10", "next_review_date": "2025-09-10", "notes": "Cayman domicile requires semi-annual review."},
    {"client_name": "Zenith Quantitative Research", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": False, "risk_tier": "LOW", "last_review_date": "2025-01-30", "next_review_date": "2026-01-30", "notes": "UK FCA regulated. Low risk profile."},
    {"client_name": "Meridian Investment Group", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": True, "risk_tier": "MEDIUM", "last_review_date": "2025-04-01", "next_review_date": "2025-10-01", "notes": "Event-driven strategy requires more frequent review."},
    {"client_name": "Vanguard Alpha Partners", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": False, "risk_tier": "MEDIUM", "last_review_date": "2025-02-15", "next_review_date": "2025-08-15", "notes": "Singapore MAS regulated. Offshore structure."},
    {"client_name": "Horizon Macro Fund", "kyc_status": "VERIFIED", "aml_status": "CLEAR", "sanctions_clear": True, "finra_registered": False, "risk_tier": "LOW", "last_review_date": "2025-03-20", "next_review_date": "2026-03-20", "notes": "Swiss FINMA regulated. Clean history."},
    {"client_name": "ACME Capital Management", "kyc_status": "EXPIRED", "aml_status": "REVIEW_NEEDED", "sanctions_clear": True, "finra_registered": True, "risk_tier": "HIGH", "last_review_date": "2024-06-15", "next_review_date": "2025-06-15", "notes": "KYC expired. Returning client needs re-verification."},
    {"client_name": "Quantum Strategies LLC", "kyc_status": "PENDING", "aml_status": "PENDING", "sanctions_clear": True, "finra_registered": True, "risk_tier": "HIGH", "last_review_date": None, "next_review_date": None, "notes": "New prospect. Full KYC/AML required."},
    {"client_name": "Pinnacle Investment Group", "kyc_status": "PENDING", "aml_status": "PENDING", "sanctions_clear": True, "finra_registered": True, "risk_tier": "MEDIUM", "last_review_date": None, "next_review_date": None, "notes": "Prospect. Initial screening needed."},
]

table_id = f'{PROJECT_ID}.{BQ_DATASET}.compliance_records'
errors = bq.insert_rows_json(table_id, COMPLIANCE)
if errors:
    print(f"❌ Errors: {errors}")
else:
    print(f"✅ Inserted {len(COMPLIANCE)} compliance records")


In [ ]:
#@title 3.5 — Populate Market Intelligence (15 Records)

INTEL = [
    {"client_name": "Atlas Global Advisors", "source_type": "SEC Filing", "title": "13F-HR Q4 2024 Filing", "summary": "Increased positions in tech sector. New holdings in AI infrastructure companies.", "relevance_score": 0.95, "date": "2025-02-15"},
    {"client_name": "Atlas Global Advisors", "source_type": "News", "title": "Atlas Expands into Asia-Pacific", "summary": "Opening new office in Tokyo. Hiring 30+ analysts for APAC coverage.", "relevance_score": 0.88, "date": "2025-03-01"},
    {"client_name": "Northstar Capital Partners", "source_type": "SEC Filing", "title": "Form ADV Annual Amendment", "summary": "AUM increased 15% YoY. Added 3 new institutional clients.", "relevance_score": 0.92, "date": "2025-01-20"},
    {"client_name": "Northstar Capital Partners", "source_type": "News", "title": "Northstar PE Fund VII Close", "summary": "Successfully closed Fund VII at $2.1B. Oversubscribed by 40%.", "relevance_score": 0.90, "date": "2025-02-28"},
    {"client_name": "Beacon Hedge Fund", "source_type": "SEC Filing", "title": "13F-HR Q4 2024 Filing", "summary": "Reduced equity exposure by 20%. Increased commodity futures positions.", "relevance_score": 0.93, "date": "2025-02-10"},
    {"client_name": "Beacon Hedge Fund", "source_type": "News", "title": "Beacon Macro Strategy Outperforms", "summary": "Fund returned +18.3% in 2024, outperforming peers by 600bps.", "relevance_score": 0.91, "date": "2025-01-15"},
    {"client_name": "Beacon Hedge Fund", "source_type": "News", "title": "Beacon Expanding to Emerging Markets", "summary": "New EM desk launching Q2 2025. Hiring senior traders from Goldman.", "relevance_score": 0.85, "date": "2025-03-05"},
    {"client_name": "Zenith Quantitative Research", "source_type": "SEC Filing", "title": "Form ADV Annual", "summary": "Systematic strategies AUM stable at $420M. Low client turnover.", "relevance_score": 0.87, "date": "2025-01-30"},
    {"client_name": "Meridian Investment Group", "source_type": "News", "title": "Meridian Event-Driven Returns", "summary": "Strong Q4 driven by M&A arbitrage positions. +12.5% annual return.", "relevance_score": 0.89, "date": "2025-01-25"},
    {"client_name": "Vanguard Alpha Partners", "source_type": "SEC Filing", "title": "13F-HR Q4 2024", "summary": "Concentrated portfolio: top 10 positions represent 65% of AUM.", "relevance_score": 0.86, "date": "2025-02-12"},
    {"client_name": "Horizon Macro Fund", "source_type": "News", "title": "Horizon Swiss Operations Review", "summary": "FINMA compliance review completed successfully. No findings.", "relevance_score": 0.84, "date": "2025-03-10"},
    {"client_name": "ACME Capital Management", "source_type": "News", "title": "ACME Capital Under New Management", "summary": "New CIO Sarah Chen appointed. Strategic pivot to systematic L/S equity.", "relevance_score": 0.94, "date": "2025-01-05"},
    {"client_name": "ACME Capital Management", "source_type": "SEC Filing", "title": "Form ADV Amendment", "summary": "Updated brochure reflecting management changes and new strategy.", "relevance_score": 0.82, "date": "2025-02-01"},
    {"client_name": "Quantum Strategies LLC", "source_type": "News", "title": "Quantum Strategies Seed Round", "summary": "Raised $50M seed from institutional allocators. Launching systematic futures fund.", "relevance_score": 0.88, "date": "2025-02-20"},
    {"client_name": "Pinnacle Investment Group", "source_type": "News", "title": "Pinnacle Quant Hiring", "summary": "Hiring data scientists and ML engineers. Building next-gen alpha platform.", "relevance_score": 0.83, "date": "2025-03-01"},
]

table_id = f'{PROJECT_ID}.{BQ_DATASET}.market_intelligence'
errors = bq.insert_rows_json(table_id, INTEL)
if errors:
    print(f"❌ Errors: {errors}")
else:
    print(f"✅ Inserted {len(INTEL)} market intelligence records")


In [ ]:
#@title 3.6 — Verify BigQuery Data

print("═" * 60)
for table in ['clients', 'compliance_records', 'market_intelligence', 'deal_packages']:
    rows = list(bq.query(f'SELECT COUNT(*) as cnt FROM `{PROJECT_ID}.{BQ_DATASET}.{table}`').result())
    print(f"  {table}: {rows[0].cnt} rows")
print("═" * 60)
print("✅ BigQuery data verified")


## 4. Clone the Project Code

Clone the Deal Desk Agent repository. If you have the code as a tarball, upload it to Colab and extract instead.

In [ ]:
#@title 4.1 — Upload or Clone Project Code

import os

# Option A: Upload a tarball
# from google.colab import files
# uploaded = files.upload()  # Upload deal-desk-agent.tar.gz
# !tar xzf deal-desk-agent.tar.gz

# Option B: If code is already in your home directory
PROJECT_DIR = os.path.expanduser("~/deal-desk-agent")

if os.path.exists(PROJECT_DIR):
    print(f"✅ Project found at {PROJECT_DIR}")
    !ls {PROJECT_DIR}
else:
    print("⚠️  Project directory not found.")
    print("   Upload deal-desk-agent.tar.gz and uncomment Option A above.")


## 5. Build & Deploy Backend to Cloud Run

The backend is a FastAPI server that handles:
- Conversational AI with Claude Sonnet 4.6 + BigQuery tools
- ADK pipeline orchestration (Research, Compliance, Risk, Synthesis)
- A2A protocol handler for Gemini Enterprise
- Salesforce browser agent trigger

In [ ]:
#@title 5.1 — Build Backend Container

!cd {PROJECT_DIR}/backend && \
gcloud builds submit \
    --tag={AR_PREFIX}/backend:latest \
    --region={INFRA_REGION} \
    --timeout=600 --quiet

print("\n✅ Backend image built")


In [ ]:
#@title 5.2 — Deploy Backend to Cloud Run

!gcloud run deploy deal-desk-backend \
    --image={AR_PREFIX}/backend:latest \
    --region={INFRA_REGION} \
    --service-account={SA_EMAIL} \
    --set-env-vars="PROJECT_ID={PROJECT_ID},REGION={REGION},BQ_DATASET={BQ_DATASET},MODEL_PROVIDER=claude,OPUS_MODEL={OPUS_MODEL},SONNET_MODEL={SONNET_MODEL},HAIKU_MODEL={HAIKU_MODEL},GOOGLE_CLOUD_PROJECT={PROJECT_ID},GOOGLE_CLOUD_LOCATION={REGION}" \
    --memory=2Gi --cpu=2 --timeout=900 \
    --max-instances=5 \
    --allow-unauthenticated \
    --quiet

BACKEND_URL = !gcloud run services describe deal-desk-backend --region={INFRA_REGION} --format='value(status.url)'
BACKEND_URL = BACKEND_URL[0]
print(f"\n✅ Backend deployed: {BACKEND_URL}")


## 6. Build & Deploy Frontend to Cloud Run

The frontend is a React application with a chat interface, agent activity feed, and embedded noVNC panel for watching the Salesforce browser agent.

In [ ]:
#@title 6.1 — Build Frontend Container

!cd {PROJECT_DIR}/frontend && \
gcloud builds submit \
    --tag={AR_PREFIX}/frontend:latest \
    --region={INFRA_REGION} \
    --timeout=600 --quiet

print("\n✅ Frontend image built")


In [ ]:
#@title 6.2 — Deploy Frontend to Cloud Run

!gcloud run deploy deal-desk-frontend \
    --image={AR_PREFIX}/frontend:latest \
    --region={INFRA_REGION} \
    --memory=512Mi \
    --allow-unauthenticated \
    --quiet

FRONTEND_URL = !gcloud run services describe deal-desk-frontend --region={INFRA_REGION} --format='value(status.url)'
FRONTEND_URL = FRONTEND_URL[0]
print(f"\n✅ Frontend deployed: {FRONTEND_URL}")


## 7. Deploy Browser VM (Computer Use)

The browser VM runs a Docker container with Xvfb (virtual display), Google Chrome, noVNC (live viewer), and the Computer Use agent server. Claude navigates Salesforce through this browser.

> **Important:** After deployment, you must manually log into Salesforce via the noVNC viewer. The browser session persists for the demo.

In [ ]:
#@title 7.1 — Build Browser VM Container

!cd {PROJECT_DIR}/computer-use && \
gcloud builds submit \
    --tag={AR_PREFIX}/computer-use:latest \
    --region={INFRA_REGION} \
    --timeout=600 --quiet

print("\n✅ Browser VM image built")


In [ ]:
#@title 7.2 — Create GCE VM with Container

!gcloud compute instances create-with-container deal-desk-browser \
    --zone={ZONE} \
    --machine-type=e2-standard-4 \
    --boot-disk-size=30GB \
    --container-image={AR_PREFIX}/computer-use:latest \
    --container-env="SALESFORCE_URL={SALESFORCE_URL},PROJECT_ID={PROJECT_ID},REGION={REGION},SONNET_MODEL={SONNET_MODEL}" \
    --scopes=cloud-platform \
    --tags=deal-desk-browser \
    --quiet 2>/dev/null || echo 'VM already exists — updating container...'

# If VM already exists, update the container
!gcloud compute instances update-container deal-desk-browser \
    --zone={ZONE} \
    --container-image={AR_PREFIX}/computer-use:latest \
    2>/dev/null || true

print("\n✅ Browser VM deployed")


In [ ]:
#@title 7.3 — Reserve Static IP & Assign to VM

# Reserve a static IP
!gcloud compute addresses create deal-desk-browser-ip --region={INFRA_REGION} 2>/dev/null || echo "Already reserved"

STATIC_IP = !gcloud compute addresses describe deal-desk-browser-ip --region={INFRA_REGION} --format='value(address)'
STATIC_IP = STATIC_IP[0]

# Check current IP
CURRENT_IP = !gcloud compute instances describe deal-desk-browser --zone={ZONE} --format='value(networkInterfaces[0].accessConfigs[0].natIP)'
CURRENT_IP = CURRENT_IP[0] if CURRENT_IP else ''

if CURRENT_IP != STATIC_IP:
    print(f"Assigning static IP {STATIC_IP} (current: {CURRENT_IP})...")
    !gcloud compute instances stop deal-desk-browser --zone={ZONE} --quiet
    !gcloud compute instances delete-access-config deal-desk-browser --zone={ZONE} --access-config-name='external-nat' 2>/dev/null || true
    !gcloud compute instances add-access-config deal-desk-browser --zone={ZONE} --address={STATIC_IP}
    !gcloud compute instances start deal-desk-browser --zone={ZONE}
else:
    print(f"Static IP already assigned: {STATIC_IP}")

print(f"\n✅ Static IP: {STATIC_IP}")


In [ ]:
#@title 7.4 — Create Firewall Rules

!gcloud compute firewall-rules create deal-desk-browser-ports \
    --allow=tcp:6080,tcp:8090 \
    --target-tags=deal-desk-browser \
    --description="noVNC (6080) + Agent Server (8090)" \
    --quiet 2>/dev/null || echo 'Firewall rule already exists'

print("✅ Firewall: ports 6080 (noVNC) and 8090 (agent server) open")


## 8. Update Backend with Browser VM IP

Set the `BROWSER_AGENT_URL` environment variable on the backend so it knows where to send Salesforce requests.

In [ ]:
#@title 8.1 — Update Backend Environment

!gcloud run services update deal-desk-backend \
    --region={INFRA_REGION} \
    --update-env-vars="BROWSER_AGENT_URL=http://{STATIC_IP}:8090" \
    --quiet

print(f"✅ Backend updated: BROWSER_AGENT_URL=http://{STATIC_IP}:8090")


## 9. Verify Deployment

Run all health checks to confirm every component is working.

In [ ]:
#@title 9.1 — Health Checks

import requests
import time

print("Waiting 90 seconds for browser VM to start...")
time.sleep(90)

print("\n═══ Health Checks ═══")

# Backend
try:
    r = requests.get(f"{BACKEND_URL}/api/health", timeout=30)
    print(f"  Backend:    ✅ {r.json()}")
except Exception as e:
    print(f"  Backend:    ❌ {e}")

# Frontend
try:
    r = requests.get(FRONTEND_URL, timeout=10)
    print(f"  Frontend:   ✅ Status {r.status_code}")
except Exception as e:
    print(f"  Frontend:   ❌ {e}")

# Browser VM - noVNC
try:
    r = requests.get(f"http://{STATIC_IP}:6080", timeout=10)
    print(f"  noVNC:      ✅ Status {r.status_code}")
except Exception as e:
    print(f"  noVNC:      ❌ {e}")

# Browser VM - Agent Server
try:
    r = requests.get(f"http://{STATIC_IP}:8090/health", timeout=10)
    print(f"  Agent:      ✅ {r.json()}")
except Exception as e:
    print(f"  Agent:      ❌ {e}")

# Quick BigQuery test
try:
    r = requests.post(f"{BACKEND_URL}/api/chat",
        json={"prompt": "show me our clients"}, timeout=60)
    print(f"  BigQuery:   ✅ Chat endpoint responding")
except Exception as e:
    print(f"  BigQuery:   ❌ {e}")

print("\n═══ URLs ═══")
print(f"  Frontend:  {FRONTEND_URL}")
print(f"  Backend:   {BACKEND_URL}")
print(f"  noVNC:     http://{STATIC_IP}:6080/vnc.html?autoconnect=true")
print(f"  Agent:     http://{STATIC_IP}:8090")
print("═" * 50)


## 10. Post-Deployment: Salesforce Login

**Manual step required:** Open the noVNC viewer and log into Salesforce.

1. Open the noVNC URL below in your browser
2. You should see Chrome with the Salesforce login page
3. Log in with your Salesforce credentials
4. Navigate to the Sales app > Opportunities
5. The browser session persists — the agent will use this authenticated session

> **Note:** If the browser shows a blank page, wait 60 seconds for Chrome to finish loading.

In [ ]:
#@title 10.1 — Open noVNC Viewer

NOVNC_URL = f"http://{STATIC_IP}:6080/vnc.html?autoconnect=true"
print(f"Open this URL in your browser to log into Salesforce:")
print(f"\n  👉 {NOVNC_URL}\n")
print("After logging in, test the full pipeline from the React UI.")


---

## 11. (Optional) Register with Gemini Enterprise via A2A

Register the Deal Desk Agent in the Gemini Enterprise Agent Gallery using the A2A (Agent-to-Agent) protocol.

**Prerequisites:**
- A Gemini Enterprise app (Standard or Plus edition)
- The Discovery Engine API enabled (done in Step 2)
- The Discovery Engine Admin role

> **Edit `GE_APP_ID` below** with your Gemini Enterprise app ID.

In [ ]:
#@title 11.1 — Configuration { display-mode: "form" }

GE_APP_ID = "your-ge-app-id"  #@param {type:"string"}

print(f"GE App ID: {GE_APP_ID}")
if GE_APP_ID == "your-ge-app-id":
    print("⚠️  Please edit GE_APP_ID above with your actual Gemini Enterprise app ID")


In [ ]:
#@title 11.2 — Register A2A Agent with Gemini Enterprise

import json
import subprocess

# Build the agent card
agent_card = json.dumps({
    "protocolVersion": "0.3.0",
    "name": "Deal Desk Agent",
    "description": "End-to-end deal desk pipeline for FSI client onboarding. Queries BigQuery for client data, runs compliance checks, scores risk, creates Salesforce opportunities via browser automation. Powered by Claude on Vertex AI + Google ADK.",
    "url": BACKEND_URL,
    "version": "1.0.0",
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text"],
    "capabilities": {"streaming": True},
    "skills": [
        {"id": "client-onboarding", "name": "Client Onboarding", "description": "Run the full deal desk pipeline", "tags": ["fsi", "onboarding"]},
        {"id": "data-query", "name": "Data Query", "description": "Query BigQuery for client records", "tags": ["bigquery"]},
        {"id": "salesforce-entry", "name": "Salesforce Entry", "description": "Create Salesforce opportunities via browser automation", "tags": ["salesforce"]},
    ]
})

# Register
TOKEN = subprocess.check_output(["gcloud", "auth", "print-access-token"]).decode().strip()
PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"]
).decode().strip()

import requests
resp = requests.post(
    f"https://global-discoveryengine.googleapis.com/v1alpha/projects/{PROJECT_ID}/locations/global/collections/default_collection/engines/{GE_APP_ID}/assistants/default_assistant/agents",
    headers={
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
        "X-Goog-User-Project": PROJECT_ID,
    },
    json={
        "displayName": "Deal Desk Agent (Conversational)",
        "description": "Conversational FSI Deal Desk agent with BigQuery data access, compliance checks, risk scoring, and Salesforce browser automation. Powered by Claude on Vertex AI.",
        "a2aAgentDefinition": {
            "jsonAgentCard": agent_card
        }
    }
)

if resp.status_code == 200:
    result = resp.json()
    AGENT_ID = result["name"].split("/")[-1]
    print(f"✅ A2A Agent registered!")
    print(f"   Agent ID: {AGENT_ID}")
    print(f"   Display Name: {result.get('displayName')}")
else:
    print(f"❌ Registration failed: {resp.status_code}")
    print(resp.text)


In [ ]:
#@title 11.3 — Grant IAM for Discovery Engine

PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format='value(projectNumber)'
PROJECT_NUMBER = PROJECT_NUMBER[0]

!gcloud run services add-iam-policy-binding deal-desk-backend \
    --region={INFRA_REGION} \
    --member="serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-discoveryengine.iam.gserviceaccount.com" \
    --role="roles/run.invoker" \
    --quiet

print(f"\n✅ Discovery Engine SA granted run.invoker on deal-desk-backend")


---

## 12. Cleanup (When Done)

**⚠️ Only run this when you are completely done with the demo.** This deletes all resources.

In [ ]:
#@title 12.1 — Delete All Resources (DESTRUCTIVE) { display-mode: "form" }

CONFIRM_DELETE = "no"  #@param ["no", "yes"]

if CONFIRM_DELETE != "yes":
    print("⚠️  Set CONFIRM_DELETE to 'yes' to proceed.")
else:
    print("🗑️ Deleting all resources...")

    # Cloud Run
    !gcloud run services delete deal-desk-backend --region={INFRA_REGION} --quiet 2>/dev/null
    !gcloud run services delete deal-desk-frontend --region={INFRA_REGION} --quiet 2>/dev/null
    print("  ✅ Cloud Run services deleted")

    # GCE VM
    !gcloud compute instances delete deal-desk-browser --zone={ZONE} --quiet 2>/dev/null
    print("  ✅ Browser VM deleted")

    # Static IP
    !gcloud compute addresses delete deal-desk-browser-ip --region={INFRA_REGION} --quiet 2>/dev/null
    print("  ✅ Static IP released")

    # Firewall
    !gcloud compute firewall-rules delete deal-desk-browser-ports --quiet 2>/dev/null
    print("  ✅ Firewall rule deleted")

    # BigQuery
    !bq rm -r -f -d {PROJECT_ID}:{BQ_DATASET} 2>/dev/null
    print("  ✅ BigQuery dataset deleted")

    # Artifact Registry images (keep repo)
    !gcloud artifacts docker images delete {AR_PREFIX}/backend --quiet --delete-tags 2>/dev/null
    !gcloud artifacts docker images delete {AR_PREFIX}/frontend --quiet --delete-tags 2>/dev/null
    !gcloud artifacts docker images delete {AR_PREFIX}/computer-use --quiet --delete-tags 2>/dev/null
    print("  ✅ Container images deleted")

    print("\n🗑️ All resources deleted.")
